In [15]:
import json
import joblib
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, classification_report
clarity_mapping = {
    'Explicit': 'Clear Reply',
    'Implicit': 'Ambivalent',
    'Dodging': "Ambivalent",
    'Deflection': "Ambivalent",
    'Partial/half-answer': "Ambivalent",
    'General': "Ambivalent",
    'Declining to answer': "Clear Non-Reply",
    'Claims ignorance': "Clear Non-Reply",
    'Clarification': "Clear Non-Reply",
}
mapping_9labels = {'Explicit': 0, 'Implicit': 1, 'Dodging': 2, 'Deflection': 3, 'Partial/half-answer': 4, 'General': 5, 'Declining to answer': 6, 'Claims ignorance': 7, 'Clarification': 8}
mapping_labels = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}



def get_data(file_path, input_data, experiment,mappinf):
    all_texts = []
    all_labels = []
    with open(file_path, "r") as f:
        data = [json.loads(line) for line in f]
        if experiment == "evasion_based_clarity":
            all_labels = [mappinf[row["evasion_label"]] for row in data]
            
        elif experiment == "direct_clarity":
            #all_labels = [mappinf[row["clarity_label"]] for row in data]
            all_labels = [mappinf[clarity_mapping[row["evasion_label"]]] for row in data]


        for row in data:
            Question = row.get('interview_question', '').strip()
            Subquestion = row.get('question', '').strip()
            if input_data == "full":
                Answer = " ".join(row.get('sentenced_answer', [])).strip()
            elif input_data == "all" or input_data == "multi":
                chosen_sentence = row.get('chosen_sentence')
                if chosen_sentence and len(chosen_sentence) > 0:
                    Answer = " ".join([item.get('sentence', '') for item in chosen_sentence]).strip()
                else:
                    Answer = " ".join(row.get('sentences', [])).strip()
            combined = f"[QUESTION] {Question}\n[ANSWER] {Answer}\n[SUBQUESTION] {Subquestion}"
            all_texts.append(combined)
    return all_texts, all_labels


In [16]:
experiments_input = ["full","all","multi"]
all_results = []
for i in experiments_input:
    if i == "full":
        train_path = "/raid/jiawen/proj_master/HF_data/runningdata/train_results_full.jsonl"
        test_path =   "/raid/jiawen/proj_master/HF_data/runningdata/test_results_full_updated.jsonl"

    elif i == "all":
        train_path = "/raid/jiawen/proj_master/HF_data/runningdata/train_results_all.jsonl"
        test_path =   "/raid/jiawen/proj_master/HF_data/runningdata/test_results_all_updated.jsonl"

    elif i == "multi":
        train_path = "/raid/jiawen/proj_master/HF_data/runningdata/train_results_multi.jsonl"
        test_path =   "/raid/jiawen/proj_master/HF_data/runningdata/test_results_multi_updated.jsonl"


In [17]:
from sklearn.model_selection import train_test_split

all_train_texts, all_train_labels = get_data(train_path, i, "evasion_based_clarity", mapping_9labels)
train_texts, val_texts, train_labels, val_labels = train_test_split(all_train_texts, all_train_labels, test_size=0.2, random_state=42, stratify=all_train_labels)  
test_texts, test_labels = get_data(test_path, i, "evasion_based_clarity", mapping_9labels)


In [18]:

mapping_9_to_3 = {
    0:0, 
    1:1,
    2:1,
    3:1,
    4:1,
    5:1,
    6:2,
    7:2,
    8:2
}
def mapping(labels9):
    y_pred_labels = [mapping_9_to_3[l] for l in labels9]
    return y_pred_labels


In [ ]:
#evasion_based
from sklearn.dummy import DummyClassifier


dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(train_texts, train_labels)

    # predict
y_pred = dummy_clf.predict(test_texts)

    # accuracy

mapped_y_true_9_to_3 = mapping(test_labels)
mapped_y_pred_9_to_3 = mapping(y_pred)

accuracy = dummy_clf.score(mapped_y_true_9_to_3, mapped_y_pred_9_to_3)
f1 = f1_score(mapped_y_true_9_to_3, mapped_y_pred_9_to_3, average='weighted') 
print("This is accuracy:", accuracy)
print("This is F1:", f1)
print(classification_report(mapped_y_true_9_to_3, mapped_y_pred_9_to_3))



This is accuracy: 1.0
This is F1: 0.10471827913688378
              precision    recall  f1-score   support

           0       0.26      1.00      0.41        79
           1       0.00      0.00      0.00       206
           2       0.00      0.00      0.00        23

    accuracy                           0.26       308
   macro avg       0.09      0.33      0.14       308
weighted avg       0.07      0.26      0.10       308



/raid/jiawen/.cache/virtualenvs/llm-nW7h6vOu-py3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/raid/jiawen/.cache/virtualenvs/llm-nW7h6vOu-py3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/raid/jiawen/.cache/virtualenvs/llm-nW7h6vOu-py3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

In [ ]:
#direct_clarity
from sklearn.dummy import DummyClassifier


dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(train_texts, train_labels)

y_pred = dummy_clf.predict(test_texts)

accuracy = dummy_clf.score(test_texts, test_labels)
f1 = f1_score(test_labels, y_pred, average='macro')  
print("This is accuracy:", accuracy)
print("This is F1:", f1)
print(classification_report(test_labels, y_pred))



This is accuracy: 0.2564935064935065
This is F1: 0.04536319265001435
              precision    recall  f1-score   support

           0       0.26      1.00      0.41        79
           1       0.00      0.00      0.00        62
           2       0.00      0.00      0.00        53
           3       0.00      0.00      0.00        17
           4       0.00      0.00      0.00         3
           5       0.00      0.00      0.00        71
           6       0.00      0.00      0.00        11
           7       0.00      0.00      0.00         8
           8       0.00      0.00      0.00         4

    accuracy                           0.26       308
   macro avg       0.03      0.11      0.05       308
weighted avg       0.07      0.26      0.10       308



/raid/jiawen/.cache/virtualenvs/llm-nW7h6vOu-py3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/raid/jiawen/.cache/virtualenvs/llm-nW7h6vOu-py3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/raid/jiawen/.cache/virtualenvs/llm-nW7h6vOu-py3.10/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.